# Nghi ngờ dữ liệu: bộ Grocery Sales có đủ tin để báo cáo không?

Giám đốc vận hành một chuỗi siêu thị gửi bạn bộ **Grocery Sales** (7 bảng, `sales.csv` 6.76 triệu dòng, 01/2018 tới
05/2018, tải từ Kaggle qua một đối tác) và xin **năm báo cáo** cho buổi họp quý:

1. Doanh thu theo tháng và theo danh mục hàng.
2. Giỏ hàng trung bình và các sản phẩm hay được mua cùng nhau.
3. Giờ cao điểm trong ngày để xếp ca thu ngân.
4. Khách hàng VIP theo thành phố.
5. Xếp hạng 23 nhân viên bán hàng để thưởng quý.

Trước khi viết dòng code phân tích nào, DA senior hỏi: **dữ liệu này có mô tả một siêu thị thật không?** Bài này đi
ngược quy trình thường lệ: bạn không tìm insight, bạn tìm **bằng chứng** (kiểm định thống kê và chart) để quyết định có
được báo cáo insight từ bộ này hay không.

`sales.csv` khoảng 517 MB: đừng `pd.read_csv` cả tệp trên laptop 4 GB. Bài dùng **DuckDB** chạy SQL thẳng trên CSV
(giới hạn bộ nhớ 1 GB), chỉ kéo về pandas các bảng đã tổng hợp vài chục dòng. Bộ dữ liệu không rõ giấy phép: chỉ dùng
cục bộ, không commit trích mẫu.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/06-data-cleaning/exercises/intermediate/02-grocery-suspicion -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, tạo VIEW DuckDB trên CSV (không nạp vào RAM)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import duckdb
import pandas as pd

GROCERY = ensure("grocery-sales-synthetic")
COLUMNS = ("{'SalesID': 'BIGINT', 'SalesPersonID': 'INTEGER', 'CustomerID': 'INTEGER', 'ProductID': 'INTEGER', "
           "'Quantity': 'INTEGER', 'Discount': 'DOUBLE', 'TotalPrice': 'DOUBLE', 'SalesDate': 'TIMESTAMP', "
           "'TransactionNumber': 'VARCHAR'}")
con = duckdb.connect()
con.execute("SET memory_limit = '1GB'")
con.execute(f"CREATE VIEW sales AS SELECT * FROM read_csv('{GROCERY / 'sales.csv'}', header = true, columns = {COLUMNS})")
for name in ["products", "categories", "customers", "cities", "countries", "employees"]:
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_csv_auto('{GROCERY / (name + '.csv')}')")
con.sql("SELECT * FROM sales LIMIT 5").df()

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

## 1. Hồ sơ bảng bán hàng bằng một câu SQL

`sales_profile(con, table="sales") -> dict` với đúng các khoá:

| Khoá | Ý nghĩa | Kiểu |
| --- | --- | --- |
| `rows` | số dòng | `int` |
| `transactions` | số `TransactionNumber` khác nhau | `int` |
| `max_lines_per_transaction` | số dòng nhiều nhất của một giao dịch | `int` |
| `total_price_zero_share` | tỉ lệ dòng có `TotalPrice = 0` | `float` |
| `missing_date_share` | tỉ lệ dòng thiếu `SalesDate` | `float` |
| `quantity_min`, `quantity_max` | `Quantity` nhỏ nhất, lớn nhất | `int` |

Tính bằng SQL trên `con` (DuckDB), chỉ kéo về một dòng kết quả.

In [ ]:
def sales_profile(con: duckdb.DuckDBPyConnection, table: str = "sales") -> dict:
    """Hồ sơ bảng bán hàng: số dòng, giao dịch, dòng/giao dịch, TotalPrice = 0, thiếu ngày, Quantity min/max."""
    row = con.execute(f"""
        SELECT count(*), count(DISTINCT TransactionNumber),
               (SELECT max(n) FROM (SELECT count(*) AS n FROM {table} GROUP BY TransactionNumber)),
               avg(CASE WHEN TotalPrice = 0 THEN 1.0 ELSE 0.0 END),
               avg(CASE WHEN SalesDate IS NULL THEN 1.0 ELSE 0.0 END),
               min(Quantity), max(Quantity)
        FROM {table}""").fetchone()
    return {
        "rows": int(row[0]), "transactions": int(row[1]), "max_lines_per_transaction": int(row[2]),
        "total_price_zero_share": float(row[3]), "missing_date_share": float(row[4]),
        "quantity_min": int(row[5]), "quantity_max": int(row[6]),
    }

In [ ]:
# explore: hồ sơ trên bộ thật (vài giây)
profile = sales_profile(con)
profile

## 2. Đếm theo một biểu thức

`count_by(con, expr, table="sales") -> pd.Series`: số dòng theo giá trị của biểu thức SQL `expr` (vd. `"Quantity"`,
`"hour(SalesDate)"`, `"CAST(SalesDate AS DATE)"`), **bỏ** các dòng có giá trị `NULL`, sắp tăng dần theo giá trị;
index là giá trị, value là số dòng.

In [ ]:
def count_by(con: duckdb.DuckDBPyConnection, expr: str, table: str = "sales") -> pd.Series:
    """Số dòng theo giá trị của biểu thức SQL expr, bỏ NULL, sắp theo giá trị."""
    df = con.execute(f"SELECT {expr} AS k, count(*) AS n FROM {table} WHERE {expr} IS NOT NULL GROUP BY 1 ORDER BY 1").df()
    return df.set_index("k")["n"]

## 3. Hai phép đo "phẳng bất thường"

**Kiểm định chi-square độ phù hợp** (goodness of fit) so số đếm quan sát với số đếm kỳ vọng nếu mọi nhóm bằng nhau.
`p_value` lớn nghĩa là **không bác bỏ được** giả thuyết "đều"; nó không chứng minh dữ liệu đều, nhưng với 6.7 triệu
dòng, dữ liệu thật hầu như luôn có `p_value` gần 0 vì chỉ cần lệch nhỏ là kiểm định bắt được. Một siêu thị thật có
`p_value` 0.5 cho số lượng mỗi dòng là chuyện không thể.

`uniformity_test(counts) -> dict`: `chi2` và `p_value` từ `scipy.stats.chisquare(counts)` (kỳ vọng đều), và
`max_min_ratio = max / min`. Giá trị là `float`.

**Chỉ số phân tán** (index of dispersion) của số đếm: phương sai chia trung bình. Đếm sự kiện độc lập, ngẫu nhiên đều
(phân phối Poisson) cho chỉ số khoảng 1. Doanh số ngày thật có mùa vụ tuần, ngày lễ, khuyến mãi nên chỉ số lên hàng
trăm.

`dispersion_index(counts) -> float`: `counts.var() / counts.mean()` (`var` mặc định của pandas, `ddof=1`).

In [ ]:
def uniformity_test(counts: pd.Series) -> dict:
    """chi2, p_value (chi-square so với phân phối đều) và max/min của số đếm."""
    res = stats.chisquare(counts.to_numpy())
    return {"chi2": float(res.statistic), "p_value": float(res.pvalue),
            "max_min_ratio": float(counts.max() / counts.min())}


def dispersion_index(counts: pd.Series) -> float:
    """Phương sai / trung bình của số đếm."""
    return float(counts.var() / counts.mean())

In [ ]:
# explore: số lượng mỗi dòng, giờ trong ngày, số dòng mỗi ngày, mỗi nhân viên, mỗi sản phẩm, mỗi khách
quantity = count_by(con, "Quantity")
hours = count_by(con, "hour(SalesDate)")
daily = count_by(con, "CAST(SalesDate AS DATE)")
checks = {
    "Quantity": quantity, "giờ": hours, "ngày": daily,
    "nhân viên": count_by(con, "SalesPersonID"), "sản phẩm": count_by(con, "ProductID"), "khách": count_by(con, "CustomerID"),
}
pd.DataFrame({name: {**uniformity_test(c), "dispersion": dispersion_index(c), "groups": len(c)} for name, c in checks.items()}).T.round(4)

In [ ]:
# explore: thứ trong tuần. Cẩn thận: 01/01-09/05/2018 có 19 thứ Hai/Ba/Tư nhưng chỉ 18 các thứ khác,
# nên đếm tổng theo thứ sẽ "không đều" dù mỗi ngày như nhau. So TRUNG BÌNH mỗi ngày theo thứ.
by_day = daily.rename_axis("d").reset_index()
by_day["weekday"] = pd.to_datetime(by_day["d"]).dt.day_name()
print(by_day.groupby("weekday")["n"].agg(["size", "mean"]).round(0))

## 4. Cờ đỏ

`red_flags(profile, quantity_test, hour_test, daily_dispersion) -> list[str]`: danh sách cờ theo **đúng thứ tự** dưới
đây, chỉ gồm các cờ bị bật:

| Cờ | Bật khi | Vì sao đáng ngờ |
| --- | --- | --- |
| `total_price_zero` | `profile["total_price_zero_share"] > 0.99` | Cột tiền gần như toàn số 0: không có doanh thu để báo cáo |
| `one_line_per_transaction` | `profile["max_lines_per_transaction"] == 1` | Siêu thị thật có giỏ nhiều món; không phân tích giỏ được |
| `uniform_quantity` | `quantity_test["p_value"] > 0.05` | Khách mua 25 món một mặt hàng nhiều ngang mua 1 món |
| `flat_hours` | `hour_test["max_min_ratio"] < 1.1` | 3 giờ sáng đông ngang 6 giờ chiều |
| `poisson_like_daily` | `daily_dispersion < 2` | Không có mùa vụ tuần, ngày lễ, khuyến mãi |

In [ ]:
FLAG_ORDER = ["total_price_zero", "one_line_per_transaction", "uniform_quantity", "flat_hours", "poisson_like_daily"]


def red_flags(profile: dict, quantity_test: dict, hour_test: dict, daily_dispersion: float) -> list[str]:
    """Các cờ đỏ bị bật, theo thứ tự FLAG_ORDER."""
    on = {
        "total_price_zero": profile["total_price_zero_share"] > 0.99,
        "one_line_per_transaction": profile["max_lines_per_transaction"] == 1,
        "uniform_quantity": quantity_test["p_value"] > 0.05,
        "flat_hours": hour_test["max_min_ratio"] < 1.1,
        "poisson_like_daily": daily_dispersion < 2,
    }
    return [flag for flag in FLAG_ORDER if on[flag]]

In [ ]:
# explore: cờ đỏ trên bộ thật
q_test, h_test, d_disp = uniformity_test(quantity), uniformity_test(hours), dispersion_index(daily)
red_flags(profile, q_test, h_test, d_disp)

## 5. Chart: "dấu vân tay" của dữ liệu sinh giả so với một siêu thị thật

Cell dưới đặt Grocery Sales cạnh **dunnhumby The Complete Journey** (giao dịch thật của 2 500 hộ ở một chuỗi siêu
thị Mỹ, bộ dùng cho capstone) nếu bạn đã tải bộ đó; chưa có thì chỉ vẽ Grocery Sales
(`uv run python data/download_dataset.py dunnhumby-complete-journey` để tải, khoảng 800 MB).

In [ ]:
# explore: 4 panel, mỗi panel một câu hỏi "siêu thị thật trông thế nào?"
DH = ROOT / "data" / "raw" / "dunnhumby-complete-journey" / "transaction_data.csv"
real = None
if DH.exists():
    dh = duckdb.connect()
    dh.execute("SET memory_limit = '1GB'")
    dh.execute(f"CREATE VIEW tx AS SELECT * FROM read_csv('{DH}')")
    real = {
        "quantity": count_by(dh, "QUANTITY", "tx").loc[1:25],
        "hours": count_by(dh, "CAST(TRANS_TIME AS INTEGER) // 100", "tx"),
        "daily": count_by(dh, "DAY", "tx"),
        "lines": dh.execute("SELECT n, count(*) FROM (SELECT count(*) AS n FROM tx GROUP BY BASKET_ID) GROUP BY 1 ORDER BY 1").df().set_index("n").iloc[:, 0],
    }

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
share = lambda s: s / s.sum()
ax[0, 0].bar(quantity.index, share(quantity), color="#e45756", label="Grocery Sales")
ax[0, 0].set_title("Quantity mỗi dòng (tỉ lệ dòng)", loc="left")
ax[0, 1].plot(hours.index, share(hours), color="#e45756", marker="o", label="Grocery Sales")
ax[0, 1].set_title("Giờ trong ngày (tỉ lệ dòng)", loc="left")
ax[1, 0].plot(range(len(daily)), daily / daily.mean(), color="#e45756", label="Grocery Sales")
ax[1, 0].set_title("Số dòng mỗi ngày / trung bình", loc="left")
ax[1, 1].bar([1], [1.0], color="#e45756", label="Grocery Sales: 100% giao dịch 1 dòng")
ax[1, 1].set_title("Số dòng mỗi giao dịch (tỉ lệ giao dịch)", loc="left")
if real:
    ax[0, 0].plot(real["quantity"].index, share(real["quantity"]), color="#4c78a8", marker=".", label="dunnhumby (thật)")
    ax[0, 1].plot(real["hours"].index, share(real["hours"]), color="#4c78a8", marker=".", label="dunnhumby (thật)")
    d = real["daily"].loc[100:600]
    ax[1, 0].plot(range(len(d)), d / d.mean(), color="#4c78a8", lw=0.8, label="dunnhumby (thật, ngày 100-600)")
    ax[1, 1].plot(real["lines"].loc[:40].index, share(real["lines"]).loc[:40], color="#4c78a8", label="dunnhumby (thật)")
for a in ax.flat:
    a.legend(frameon=False, fontsize=8)
    a.spines[["top", "right"]].set_visible(False)
fig.suptitle("Grocery Sales phẳng ở mọi chiều mà một siêu thị thật luôn gồ ghề", x=0.01, ha="left")
fig.tight_layout()
plt.show()

In [ ]:
# explore: các bảng chiều cũng kể cùng câu chuyện
print(con.sql("SELECT * FROM countries WHERE CountryName IN ('United States', 'Canada', 'Germany', 'Vietnam')").df())
print(con.sql("SELECT CountryID, count(*) AS cities FROM cities GROUP BY 1").df())
print(con.sql("SELECT p.ProductName, c.CategoryName, round(p.Price, 2) AS price FROM products p JOIN categories c USING (CategoryID) "
              "WHERE p.ProductName LIKE 'Wine%' OR p.ProductName LIKE 'Beef%' OR p.ProductName LIKE 'Halibut%' LIMIT 8").df())
print(con.sql("SELECT min(n), max(n) FROM (SELECT CityID, count(*) AS n FROM customers GROUP BY 1)").df())

## 6. Kết luận cho Giám đốc vận hành, rồi làm trắc nghiệm

Với từng báo cáo trong 5 yêu cầu, ghi: cột nào cần, bằng chứng nào cho thấy cột đó không mô tả hành vi thật, và
**điều bạn sẽ trả lời** (không phải điều bạn sẽ vẽ). Một chart "giờ cao điểm" vẽ trên dữ liệu phẳng vẫn có một giờ cao
nhất, và ai đó sẽ xếp ca theo nó.

Viết câu kết luận một dòng, như dòng đầu email trả lời. Xong thì mở app, làm phần **trắc nghiệm** của bài.